# tiny-ton on a Real GPU

This notebook builds **tiny-ton** from source with CUDA enabled and runs `vector_add` on the Colab T4 GPU.

**Before running:** Go to *Runtime > Change runtime type* and select **T4 GPU**.

## 1. Install LLVM/MLIR 18 + pybind11

Uses pre-built packages from `apt.llvm.org` (~2 min).

In [ ]:
%%bash
set -e
echo '>>> Adding LLVM 18 apt repository...'
wget -qO- https://apt.llvm.org/llvm-snapshot.gpg.key | tee /etc/apt/trusted.gpg.d/apt.llvm.org.asc > /dev/null
add-apt-repository -y 'deb http://apt.llvm.org/jammy/ llvm-toolchain-jammy-18 main' > /dev/null 2>&1
echo '>>> Installing packages...'
apt-get update -qq > /dev/null 2>&1
apt-get install -y -qq libmlir-18-dev mlir-18-tools llvm-18-dev cmake ninja-build > /dev/null 2>&1
pip install -q pybind11
echo '>>> Done. MLIR version:'
mlir-opt-18 --version 2>&1 | head -2

## 2. Clone tiny-ton

In [ ]:
%%bash
set -e
if [ ! -d /content/tiny-ton ]; then
  git clone https://github.com/ganeshnj/tiny-ton.git /content/tiny-ton
else
  echo 'tiny-ton already cloned, pulling latest...'
  cd /content/tiny-ton && git pull
fi

## 3. Build with CUDA enabled

Compiles the C++ compiler, MLIR lowering passes, CUDA runtime, and Python bindings (~1-2 min).

In [ ]:
%%bash
set -e
cd /content/tiny-ton
rm -rf build
cmake -G Ninja -S . -B build \
  -DCMAKE_BUILD_TYPE=Release \
  -DMLIR_DIR=/usr/lib/llvm-18/lib/cmake/mlir \
  -DLLVM_DIR=/usr/lib/llvm-18/lib/cmake/llvm \
  -DTTN_ENABLE_CUDA=ON \
  -DTTN_ENABLE_PYTHON=ON \
  -DCUDAToolkit_ROOT=/usr/local/cuda \
  -Dpybind11_DIR=$(python3 -c 'import pybind11; print(pybind11.get_cmake_dir())')
cmake --build build
echo '>>> Build complete.'

## 4. Verify GPU

In [ ]:
import sys, os
sys.path.insert(0, '/content/tiny-ton/build/bindings')
import importlib
importlib.invalidate_caches()
import _tiny_ton_core as core
print(f'has_cuda() = {core.has_cuda()}')

## 5. Debug pipeline (stage-by-stage IR dump)

Shows every compilation stage: Python source -> AST -> TinyTon MLIR -> simulator assembly -> PTX.

In [ ]:
import os
os.environ['TTN_SM_VERSION'] = 'sm_75'

!cd /content/tiny-ton && TTN_SM_VERSION=sm_75 \
  PYTHONPATH=build/bindings:python \
  python3 examples/debug_pipeline.py

## 6. CPU vs GPU — Setup

Import TinyTon and define a helper that compares a CPU (numpy) result against a GPU (TinyTon kernel) result.

In [ ]:
import sys, os
sys.path.insert(0, '/content/tiny-ton/build/bindings')
sys.path.insert(0, '/content/tiny-ton/python')
os.environ['TTN_SM_VERSION'] = 'sm_75'

import numpy as np
import tiny_ton as tt

results = []

def compare(name, cpu, gpu, atol=1e-5):
    ok = np.allclose(cpu, gpu, atol=atol, rtol=1e-2)
    tag = "PASS" if ok else "FAIL"
    cpu_val = cpu.flat[0]
    gpu_val = gpu.flat[0]
    print(f"  {name:24s}  cpu={cpu_val:12.6f}  gpu={gpu_val:12.6f}  {tag}")
    if not ok:
        print(f"  {'':24s}  max_diff={np.max(np.abs(cpu.astype(np.float64) - gpu.astype(np.float64))):.6e}")
    results.append((name, ok))
    return ok

def compare_exact(name, cpu, gpu):
    ok = np.array_equal(cpu, gpu)
    tag = "PASS" if ok else "FAIL"
    print(f"  {name:24s}  cpu={cpu.flat[0]:12d}  gpu={gpu.flat[0]:12d}  {tag}")
    if not ok:
        mismatches = np.sum(cpu != gpu)
        print(f"  {'':24s}  {mismatches} mismatches out of {cpu.size}")
    results.append((name, ok))
    return ok

print("helpers loaded.")

## 7. CPU vs GPU — vector_add (i32, f32, f16)

For each dtype: compute `a + b` on the CPU with numpy, then run the same operation through the TinyTon GPU kernel, and compare.

In [ ]:
@tt.jit
def vector_add(a_ptr, b_ptr, c_ptr, N):
    pid = tt.program_id(0)
    offsets = pid * 64 + tt.arange(0, 64)
    mask = offsets < N
    a = tt.load(a_ptr + offsets, mask=mask)
    b = tt.load(b_ptr + offsets, mask=mask)
    tt.store(c_ptr + offsets, a + b, mask=mask)

N = 256
grid = (N // 64,)
np.random.seed(0)

for dtype_name, dtype, atol in [('i32', np.int32, 0),
                                 ('f32', np.float32, 1e-6),
                                 ('f16', np.float16, 1e-2)]:
    print(f'\n--- vector_add {dtype_name} ---')
    if dtype == np.int32:
        a = np.arange(N, dtype=dtype)
        b = np.arange(N, dtype=dtype)
    else:
        a = np.random.randn(N).astype(dtype)
        b = np.random.randn(N).astype(dtype)

    cpu_c = (a + b).astype(dtype)

    gpu_c = np.zeros(N, dtype=dtype)
    vector_add[grid](a.copy(), b.copy(), gpu_c, N)

    if dtype == np.int32:
        compare_exact(f'add_{dtype_name}', cpu_c, gpu_c)
    else:
        compare(f'add_{dtype_name}', cpu_c, gpu_c, atol=atol)

## 8. CPU vs GPU — Math intrinsics (exp, log, sqrt, rsqrt, abs, max)

For each intrinsic: compute on CPU with numpy, then run through the TinyTon GPU kernel, and compare.

In [ ]:
@tt.jit
def kern_exp(src, dst, N):
    pid = tt.program_id(0)
    off = pid * 64 + tt.arange(0, 64)
    mask = off < N
    tt.store(dst + off, tt.exp(tt.load(src + off, mask=mask)), mask=mask)

@tt.jit
def kern_log(src, dst, N):
    pid = tt.program_id(0)
    off = pid * 64 + tt.arange(0, 64)
    mask = off < N
    tt.store(dst + off, tt.log(tt.load(src + off, mask=mask)), mask=mask)

@tt.jit
def kern_sqrt(src, dst, N):
    pid = tt.program_id(0)
    off = pid * 64 + tt.arange(0, 64)
    mask = off < N
    tt.store(dst + off, tt.sqrt(tt.load(src + off, mask=mask)), mask=mask)

@tt.jit
def kern_rsqrt(src, dst, N):
    pid = tt.program_id(0)
    off = pid * 64 + tt.arange(0, 64)
    mask = off < N
    tt.store(dst + off, tt.rsqrt(tt.load(src + off, mask=mask)), mask=mask)

@tt.jit
def kern_abs(src, dst, N):
    pid = tt.program_id(0)
    off = pid * 64 + tt.arange(0, 64)
    mask = off < N
    tt.store(dst + off, tt.abs(tt.load(src + off, mask=mask)), mask=mask)

@tt.jit
def kern_max(a_ptr, b_ptr, dst, N):
    pid = tt.program_id(0)
    off = pid * 64 + tt.arange(0, 64)
    mask = off < N
    a = tt.load(a_ptr + off, mask=mask)
    b = tt.load(b_ptr + off, mask=mask)
    tt.store(dst + off, tt.max(a, b), mask=mask)

N = 256
grid = (N // 64,)
np.random.seed(42)

for dtype_name, dtype, atol in [('f32', np.float32, 1e-5), ('f16', np.float16, 5e-2)]:
    print(f'\n--- math intrinsics {dtype_name} ---')
    pos = (np.abs(np.random.randn(N)) + 0.01).astype(dtype)
    mixed = (np.random.randn(N) * 2).astype(dtype)

    for name, kernel, np_fn, inp in [
        ('exp',  kern_exp,  np.exp,  mixed),
        ('log',  kern_log,  np.log,  pos),
        ('sqrt', kern_sqrt, np.sqrt, pos),
        ('abs',  kern_abs,  np.abs,  mixed),
    ]:
        cpu_out = np_fn(inp).astype(dtype)
        gpu_out = np.zeros(N, dtype=dtype)
        kernel[grid](inp.copy(), gpu_out, N)
        compare(f'{name}_{dtype_name}', cpu_out, gpu_out, atol=atol)

    cpu_rsqrt = (1.0 / np.sqrt(pos.astype(np.float32))).astype(dtype)
    gpu_rsqrt = np.zeros(N, dtype=dtype)
    kern_rsqrt[grid](pos.copy(), gpu_rsqrt, N)
    compare(f'rsqrt_{dtype_name}', cpu_rsqrt, gpu_rsqrt, atol=atol)

    a = (np.random.randn(N) * 5).astype(dtype)
    b = (np.random.randn(N) * 5).astype(dtype)
    cpu_max = np.maximum(a, b)
    gpu_max = np.zeros(N, dtype=dtype)
    kern_max[grid](a.copy(), b.copy(), gpu_max, N)
    compare(f'max_{dtype_name}', cpu_max, gpu_max, atol=atol)

## 9. CPU vs GPU — Reductions (reduce_sum, reduce_max)

For each block: compute the reduction on CPU with numpy, then run through the TinyTon GPU kernel (warp-shuffle + shared memory), and compare.

In [ ]:
@tt.jit
def kern_reduce_sum(src, dst, N):
    pid = tt.program_id(0)
    off = pid * 64 + tt.arange(0, 64)
    mask = off < N
    x = tt.load(src + off, mask=mask)
    total = tt.reduce_sum(x)
    tt.store(dst + pid, total)

@tt.jit
def kern_reduce_max(src, dst, N):
    pid = tt.program_id(0)
    off = pid * 64 + tt.arange(0, 64)
    mask = off < N
    x = tt.load(src + off, mask=mask)
    mx = tt.reduce_max(x)
    tt.store(dst + pid, mx)

N = 256
BLOCK = 64
grid = (N // BLOCK,)
np.random.seed(123)

for dtype_name, dtype, atol in [('f32', np.float32, 1e-4)]:
    print(f'\n--- reductions {dtype_name} ---')
    x = np.random.randn(N).astype(dtype)

    cpu_sums = np.array([np.sum(x[i*BLOCK:(i+1)*BLOCK]) for i in range(N // BLOCK)], dtype=dtype)
    gpu_sums = np.zeros(N // BLOCK, dtype=dtype)
    kern_reduce_sum[grid](x.copy(), gpu_sums, N)
    for i in range(N // BLOCK):
        compare(f'reduce_sum_blk{i}_{dtype_name}',
                cpu_sums[i:i+1], gpu_sums[i:i+1], atol=atol)

    cpu_maxs = np.array([np.max(x[i*BLOCK:(i+1)*BLOCK]) for i in range(N // BLOCK)], dtype=dtype)
    gpu_maxs = np.zeros(N // BLOCK, dtype=dtype)
    kern_reduce_max[grid](x.copy(), gpu_maxs, N)
    for i in range(N // BLOCK):
        compare(f'reduce_max_blk{i}_{dtype_name}',
                cpu_maxs[i:i+1], gpu_maxs[i:i+1], atol=atol)

## 10. CPU vs GPU — ReLU (Rectified Linear Unit)

Element-wise `max(x, 0)`. Composed from existing `tt.max` — no new C++ ops.

In [ ]:
@tt.jit
def kern_relu(src, dst, N):
    pid = tt.program_id(0)
    off = pid * 64 + tt.arange(0, 64)
    mask = off < N
    x = tt.load(src + off, mask=mask)
    y = tt.relu(x)
    tt.store(dst + off, y, mask=mask)

N = 256
grid = (N // 64,)
np.random.seed(99)

for dtype_name, dtype, atol in [('f32', np.float32, 0.0), ('f16', np.float16, 0.0)]:
    print(f'\n--- relu {dtype_name} ---')
    x = (np.random.randn(N) * 5).astype(dtype)
    cpu_relu = np.maximum(x, dtype(0))
    gpu_relu = np.zeros(N, dtype=dtype)
    kern_relu[grid](x.copy(), gpu_relu, N)
    compare(f'relu_{dtype_name}', cpu_relu, gpu_relu, atol=atol)

## 11. Summary

In [ ]:
passed = sum(1 for _, ok in results if ok)
failed = sum(1 for _, ok in results if not ok)
total = passed + failed

print(f'\n{"="*60}')
print(f'  CPU vs GPU results:  {passed}/{total} passed, {failed} failed')
print(f'{"="*60}')

if failed:
    print('\n  Failed tests:')
    for name, ok in results:
        if not ok:
            print(f'    - {name}')

assert failed == 0, f'{failed} test(s) failed!'
print('\n  All CPU vs GPU tests passed!')